# Normalized planar deposition from point and disk sources

## Prerequisites

See the [planar deposition geometry lecture note](../../../docs/lecture-notes/deposition/planar-source-geometry/index.md).

This laboratory assumes elementary vector geometry, inverse-square spreading, surface projection, numerical quadrature, and the distinction between an isotropic point source and a cosine-power surface source.

## Objectives

1. Evaluate the closed-form normalized profile of an isotropic 3D point source incident on a parallel 2D substrate.
2. Integrate finite circular source profiles in polar coordinates.
3. Distinguish isotropic ($n=0$) and Lambertian ($n=1$) surface emission.
4. Check the finite-disk quadrature against refinement and the isotropic point-source limit.

Only normalized geometric profiles are evaluated. Vapor pressure, Hertz--Knudsen emission, material density, sticking, re-evaporation, and chamber transport are outside this laboratory.


## Geometric models

Place the substrate in $z=0$ and a point source at $(0,0,-h)$. For lateral offset $\ell$, the source-to-substrate distance is $r=(h^2+\ell^2)^{1/2}$ and the substrate projection factor is $h/r$. Isotropic inverse-square spreading therefore gives

$$
\frac{d(\ell)}{d(0)}
=\frac{h^3}{(h^2+\ell^2)^{3/2}}
=\left[1+(\ell/h)^2\right]^{-3/2}.
$$

For a circular source of radius $a$ with emission intensity proportional to $\cos^n\phi$, the unnormalized shape is

$$
J_n(\ell)\propto
\int_0^a\int_0^{2\pi}
\frac{h^{n+1}\rho\,d\psi\,d\rho}
{\left(\ell^2+\rho^2-2\ell\rho\cos\psi+h^2\right)^{(n+3)/2}}.
$$

The maintained model evaluates the dimensionless form and returns $J_n(\ell)/J_n(0)$. As $a/h\to0$, the $n=0$ disk approaches the isotropic point-source profile. The $n=1$ limit is instead proportional to $[1+(\ell/h)^2]^{-2}$ because its angular emission law is Lambertian.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.deposition.geometry.plane_point import (
    PlanarPointSource3DDepositionModel,
)
from projectkoios.physkit.deposition.surface_source.disk import (
    DiskSourcePolarQuadrature,
    PlanarDiskSource3DDepositionModel,
)
from projectkoios.physkit.units import (
    PhysicalUnit,
    ScalarQuantity,
    Unitless,
    VectorQuantity,
)

In [ ]:
length_unit = PhysicalUnit(expression="meter")
source_to_substrate_distance = ScalarQuantity(magnitude=0.2, unit=length_unit)
source_radius = ScalarQuantity(magnitude=0.1, unit=length_unit)
lateral_offsets = VectorQuantity(
    magnitude=np.linspace(0.0, 0.4, 161, dtype=np.float64),
    unit=length_unit,
)
dimensionless_offsets = (
    lateral_offsets.magnitude / source_to_substrate_distance.magnitude
)

point_model = PlanarPointSource3DDepositionModel(
    source_to_substrate_distance=source_to_substrate_distance,
)
point_profile = point_model.evaluate_normalized_profile(
    lateral_offsets=lateral_offsets,
)

isotropic_disk_model = PlanarDiskSource3DDepositionModel(
    source_to_substrate_distance=source_to_substrate_distance,
    source_radius=source_radius,
    cosine_exponent=ScalarQuantity(magnitude=0.0, unit=Unitless()),
)
lambertian_disk_model = PlanarDiskSource3DDepositionModel(
    source_to_substrate_distance=source_to_substrate_distance,
    source_radius=source_radius,
    cosine_exponent=ScalarQuantity(magnitude=1.0, unit=Unitless()),
)
fine_quadrature = DiskSourcePolarQuadrature(
    radial_point_count=161,
    azimuthal_point_count=240,
)
isotropic_disk_profile = isotropic_disk_model.evaluate_normalized_profile(
    lateral_offsets=lateral_offsets,
    quadrature=fine_quadrature,
)
lambertian_disk_profile = lambertian_disk_model.evaluate_normalized_profile(
    lateral_offsets=lateral_offsets,
    quadrature=fine_quadrature,
)

In [ ]:
closed_form_point_profile = (1.0 + dimensionless_offsets**2) ** (-1.5)
assert np.allclose(
    point_profile.normalized_thickness.magnitude,
    closed_form_point_profile,
    rtol=1.0e-14,
    atol=0.0,
)
for profile in (isotropic_disk_profile, lambertian_disk_profile):
    assert np.isclose(profile.normalized_thickness.magnitude[0], 1.0)
    assert np.all(np.diff(profile.normalized_thickness.magnitude) < 0.0)

coarse_quadrature = DiskSourcePolarQuadrature(
    radial_point_count=81,
    azimuthal_point_count=120,
)
coarse_lambertian_profile = lambertian_disk_model.evaluate_normalized_profile(
    lateral_offsets=lateral_offsets,
    quadrature=coarse_quadrature,
)
quadrature_relative_difference = (
    np.abs(
        coarse_lambertian_profile.normalized_thickness.magnitude
        - lambertian_disk_profile.normalized_thickness.magnitude
    )
    / lambertian_disk_profile.normalized_thickness.magnitude
)
maximum_quadrature_relative_difference = float(np.max(quadrature_relative_difference))
assert maximum_quadrature_relative_difference < 1.0e-4
maximum_quadrature_relative_difference

In [ ]:
figure, (profile_axis, convergence_axis) = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(12.0, 4.5),
)
profile_axis.plot(
    dimensionless_offsets,
    point_profile.normalized_thickness.magnitude,
    label="isotropic point source",
)
profile_axis.plot(
    dimensionless_offsets,
    isotropic_disk_profile.normalized_thickness.magnitude,
    label="isotropic disk, a/h = 0.5",
)
profile_axis.plot(
    dimensionless_offsets,
    lambertian_disk_profile.normalized_thickness.magnitude,
    label="Lambertian disk, a/h = 0.5",
)
profile_axis.set(
    xlabel="lateral offset, ℓ/h",
    ylabel="normalized thickness, d/d(0)",
    title="Parallel-plane deposition profiles",
)
profile_axis.grid(True)
profile_axis.legend()

convergence_axis.semilogy(
    dimensionless_offsets,
    quadrature_relative_difference,
    label="81×120 versus 161×240",
)
convergence_axis.set(
    xlabel="lateral offset, ℓ/h",
    ylabel="relative profile difference",
    title="Lambertian-disk quadrature refinement",
)
convergence_axis.grid(True)
convergence_axis.legend()
figure.tight_layout()
plt.show()

## Interpretation

Finite source area broadens the on-axis neighborhood relative to the corresponding point-source limit. Changing the cosine exponent changes the source angular law, so a Lambertian shrinking disk does not approach the isotropic point-source exponent used here.

These curves verify the stated idealized geometry and quadrature behavior only. They do not establish a physical deposition rate or validate a particular deposition apparatus.

## Exercises

1. Vary $a/h$ and identify how the normalized profile changes.
2. Verify numerically that a shrinking $n=1$ disk approaches $[1+(\ell/h)^2]^{-2}$.
3. Repeat the quadrature refinement at larger $a/h$.
4. List the additional physical inputs needed to turn a normalized profile into a predicted film-thickness rate.
